In [31]:
import pandas as pd
from silver_staging_utils import connect_to_postgres, read_table
import numpy as np
import json
import re
import unicodedata

# Productos

In [32]:
query = '''
SELECT
    *
FROM silver.products
WHERE
    snapshot_date IN (
        SELECT MAX(snapshot_date) FROM silver.products  
    )
'''
print(query)

conn = connect_to_postgres()
if conn:
    df = read_table(query, conn)
    conn.close()


SELECT
    *
FROM silver.products
WHERE
    snapshot_date IN (
        SELECT MAX(snapshot_date) FROM silver.products  
    )

✅ Conectado a PostgreSQL


/workspaces/tesis-ivan-gennaro/scripts/silver_staging/silver_staging_utils.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45260 entries, 0 to 45259
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   snapshot_date    45260 non-null  object        
 1   supermarket      45260 non-null  object        
 2   product_id       16880 non-null  object        
 3   product_name     45260 non-null  object        
 4   brand            15990 non-null  object        
 5   price            45260 non-null  object        
 6   unit_of_measure  23476 non-null  object        
 7   is_on_promotion  6770 non-null   object        
 8   promotion_price  9045 non-null   object        
 9   category_slug    45260 non-null  object        
 10  ingestion_time   45260 non-null  datetime64[ns]
 11  created_at       45260 non-null  datetime64[ns]
dtypes: datetime64[ns](2), object(10)
memory usage: 4.1+ MB


In [34]:
df['snapshot_date'].unique()

array([datetime.date(2025, 10, 26)], dtype=object)

In [35]:
df['supermarket'].unique()

array(['real', 'stock', 'casarica', 'biggie'], dtype=object)

## Final Price creation

##### Price problem with Casa Rica

In [36]:
df["price"] = (
    df["price"]
    .astype(str)
    .str.replace(r"[^\d,\.]", "", regex=True)
    .str.replace(".", "", regex=False) 
    .str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

##### Coalesce to build final price

In [37]:
df.loc[df['promotion_price'] == '0', 'promotion_price'] = None

In [38]:
df["final_price"] = (
    df["promotion_price"]
        .combine_first(df['price'])
)

##### Validations

In [39]:
df.loc[df['supermarket'] == 'casarica', 'price'].head()

26816    120000
26817     32500
26818     23000
26819    115000
26820     59950
Name: price, dtype: int64

In [40]:
df['promotion_price'].unique()

array([None, '4500.0', '2400.0', ..., '18000', '1250', '15500'],
      shape=(1372,), dtype=object)

In [41]:
# Casa Rica get numeric
df.loc[df['supermarket'] == 'casarica', ['price', 'promotion_price', 'final_price']]

,price,promotion_price,final_price
26816,120000,None,120000
26817,32500,None,32500
26818,23000,None,23000
26819,115000,None,115000
26820,59950,None,59950
...,...,...,...
38485,27600,None,27600
38486,8700,None,8700
38487,46250,None,46250
38488,46250,None,46250


In [42]:
# Coalesce
df.loc[(df['supermarket'] == 'biggie') & (df['promotion_price'].notna()), ['price', 'promotion_price', 'final_price']]

,price,promotion_price,final_price
38638,40900,36900,36900
38642,14850,12500,12500
38644,26900,22900,22900
38645,13900,11500,11500
38647,23900,20350,20350
...,...,...,...
45247,3500,3150,3150
45249,15500,14250,14250
45250,20000,16000,16000
45251,20000,16000,16000


In [43]:
df.loc[(df['supermarket'] == 'real') & (df['promotion_price'].notna()), ['price', 'promotion_price', 'final_price']]

,price,promotion_price,final_price
8,5000,4500.0,4500.0
9,5000,4500.0,4500.0
10,2650,2400.0,2400.0
11,2650,2400.0,2400.0
12,18950,17000.0,17000.0
...,...,...,...
10036,19050,17050.0,17050.0
10056,55000,43500.0,43500.0
10069,10200,9100.0,9100.0
10070,21550,19200.0,19200.0


In [44]:
df.head()

,snapshot_date,supermarket,product_id,product_name,brand,price,unit_of_measure,is_on_promotion,promotion_price,category_slug,ingestion_time,created_at,final_price
0,2025-10-26,real,7840005011273,"Yogur con cereal Zucosos Trebol, 150 gr",Trebol,3550,None,None,None,Cat6,2025-10-26 10:15:08.851781,2025-10-27 00:14:09.900636,3550
1,2025-10-26,real,8076800195019,"Fideo Barilla capellini, 500 grs",Barilla,25900,None,None,None,Cat1,2025-10-26 10:15:08.851781,2025-10-27 00:14:09.900636,25900
2,2025-10-26,real,7896035911908,"Farofa Amafil tradicional, 250 grs",Amafil,8200,None,None,None,Cat1,2025-10-26 10:15:08.851781,2025-10-27 00:14:09.900636,8200
3,2025-10-26,real,8076800195057,"Fideo Barilla spaghetti, 500 grs",Barilla,25900,None,None,None,Cat1,2025-10-26 10:15:08.851781,2025-10-27 00:14:09.900636,25900
4,2025-10-26,real,8076802085738,"Fideo Barilla penne, 500 grs",Barilla,30900,None,None,None,Cat1,2025-10-26 10:15:08.851781,2025-10-27 00:14:09.900636,30900


# Categoria

In [45]:
query = '''
SELECT
    *
FROM silver.categories
WHERE
    snapshot_date IN (
        SELECT MAX(snapshot_date) FROM silver.categories  
    )
'''
print(query)

conn = connect_to_postgres()
if conn:
    df = read_table(query, conn)
    conn.close()


SELECT
    *
FROM silver.categories
WHERE
    snapshot_date IN (
        SELECT MAX(snapshot_date) FROM silver.categories  
    )

✅ Conectado a PostgreSQL


/workspaces/tesis-ivan-gennaro/scripts/silver_staging/silver_staging_utils.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1606 entries, 0 to 1605
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   snapshot_date       1606 non-null   object        
 1   supermarket         1606 non-null   object        
 2   category_lvl1_name  1606 non-null   object        
 3   category_lvl2_name  1561 non-null   object        
 4   category_lvl3_name  1236 non-null   object        
 5   category_lvl1_id    45 non-null     object        
 6   category_lvl2_id    0 non-null      object        
 7   category_lvl3_id    0 non-null      object        
 8   category_lvl1_slug  45 non-null     object        
 9   category_lvl2_slug  325 non-null    object        
 10  category_lvl3_slug  1236 non-null   object        
 11  created_at          1606 non-null   datetime64[ns]
dtypes: datetime64[ns](1), object(11)
memory usage: 150.7+ KB


In [47]:
df['snapshot_date'].unique()

array([datetime.date(2025, 10, 26)], dtype=object)

In [48]:
df['supermarket'].unique()

array(['biggie', 'real', 'casa_rica', 'stock'], dtype=object)

### category_final_slug
Creacion de un slug homogeneo para todos los supermercados, a utilizar para la surrogate key

In [49]:
mask_real = df['supermarket'] == 'real'

df.loc[mask_real, 'real_lvl1_clean'] = (
    df.loc[mask_real, 'category_lvl1_slug']
        .str.split('/', n=1)
        .str[-1]
)


In [50]:
df['category_slug_final'] = (
    np.where(df['supermarket'] == 'biggie', df['category_lvl1_slug'],
    np.where(df['supermarket'] == 'casa rica', df['category_lvl2_slug'],
    np.where(df['supermarket'].isin(['stock', 'super seis']), df['category_lvl3_slug'],
    np.where(df['supermarket'] == 'real', df['real_lvl1_clean'],
             None))))
)

In [51]:
df.head(1000)

,snapshot_date,supermarket,category_lvl1_name,category_lvl2_name,category_lvl3_name,category_lvl1_id,category_lvl2_id,category_lvl3_id,category_lvl1_slug,category_lvl2_slug,category_lvl3_slug,created_at,real_lvl1_clean,category_slug_final
0,2025-10-26,biggie,Alimentos Especiales,None,None,6,None,None,alimentos-especiales,None,None,2025-10-27 00:10:31.942428,NaN,alimentos-especiales
1,2025-10-26,biggie,Almacén,None,None,1,None,None,almacen,None,None,2025-10-27 00:10:31.942428,NaN,almacen
2,2025-10-26,biggie,Asado,None,None,246,None,None,asado,None,None,2025-10-27 00:10:31.942428,NaN,asado
3,2025-10-26,biggie,Bebes,None,None,41,None,None,bebes,None,None,2025-10-27 00:10:31.942428,NaN,bebes
4,2025-10-26,biggie,Bebidas con Alcohol,None,None,3,None,None,bebidas-con-alcohol,None,None,2025-10-27 00:10:31.942428,NaN,bebidas-con-alcohol
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2025-10-26,stock,Almacén,Alimentos Secos,Legumbres secas,None,None,None,None,None,Almacén/Alimentos_Secos/Legumbres_secas,2025-10-27 00:10:35.898258,NaN,Almacén/Alimentos_Secos/Legumbres_secas
996,2025-10-26,stock,Almacén,Alimentos Secos,Leche en Polvo/varios,None,None,None,None,None,Almacén/Alimentos_Secos/Leche_en_Polvo/varios,2025-10-27 00:10:35.898258,NaN,Almacén/Alimentos_Secos/Leche_en_Polvo/varios
997,2025-10-26,stock,Almacén,Alimentos Secos,Postres en Polvo,None,None,None,None,None,Almacén/Alimentos_Secos/Postres_en_Polvo,2025-10-27 00:10:35.898258,NaN,Almacén/Alimentos_Secos/Postres_en_Polvo
998,2025-10-26,stock,Almacén,Alimentos Secos,Repostería/ Pasteleria,None,None,None,None,None,Almacén/Alimentos_Secos/Repostería/_Pasteleria,2025-10-27 00:10:35.898258,NaN,Almacén/Alimentos_Secos/Repostería/_Pasteleria


### category_final_name

Análisis

In [52]:
query = '''
SELECT
    supermarket,
	category_lvl1_name
FROM silver.categories
'''
print(query)

conn = connect_to_postgres()
if conn:
    df = read_table(query, conn)
    conn.close()


SELECT
    supermarket,
	category_lvl1_name
FROM silver.categories

✅ Conectado a PostgreSQL


/workspaces/tesis-ivan-gennaro/scripts/silver_staging/silver_staging_utils.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [53]:
df.head()

,supermarket,category_lvl1_name
0,biggie,Alimentos Especiales
1,biggie,Almacén
2,biggie,Asado
3,biggie,Bebes
4,biggie,Bebidas con Alcohol


In [54]:
pivot = (
    df
    .assign(value=True)          # Marcamos presencia
    .pivot_table(
        index='category_lvl1_name',
        columns='supermarket',
        values='value',
        aggfunc='any',           # Si existe al menos 1 → True
        fill_value=False
    )
)

pivot_int = pivot.astype(int)


In [ ]:
# pivot_int.to_csv("/workspaces/tesis-ivan-gennaro/scripts/silver_staging/category_final_name_analisis.csv")

Aplicacion

In [62]:
def normalize_text(s):
    """
    Normaliza una cadena para matching:
      - pasa a minúsculas
      - quita acentos
      - reemplaza caracteres no alfanuméricos por espacio
      - reduce espacios múltiples a uno
      - strip()
    """
    if s is None:
        return None
    s = str(s).strip().lower()
    # quitar acentos
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    # deja solo letras, números y espacios
    s = re.sub(r"[^a-z0-9\s]+", " ", s)
    # colapsar espacios
    s = re.sub(r"\s+", " ", s).strip()
    return s


In [63]:
df['category_clean'] = df['category_lvl1_name'].apply(normalize_text)


In [64]:

MAPPINGS_FILE = '/workspaces/tesis-ivan-gennaro/scripts/silver_staging/mapeo_categoria_final.json'

In [65]:
# Leer mappings desde JSON
with open(MAPPINGS_FILE, 'r', encoding='utf-8') as f:
    mappings = json.load(f)

inv_maps = {sup: {} for sup in ["biggie", "casa_rica", "real", "s6", "stock"]}

for final, mapping in mappings.items():
    for sup, cats in mapping.items():
        if cats is None:
            continue
        
        # Normalizar: convertir string -> lista de strings
        if isinstance(cats, str):
            cats = [cats]

        # Normalizar cada valor
        cats_norm = [normalize_text(c) for c in cats]

        # Guardar mapping categoria_super → categoria_final
        for c in cats_norm:
            inv_maps[sup][c] = final   # final NO normalizado, mejor para presentación


In [66]:
# 3) Mapear la categoría final
df['category_final'] = df.apply(
    lambda row: inv_maps[row['supermarket']].get(row['category_clean']),
    axis=1
)

In [67]:
df.head()

,supermarket,category_lvl1_name,category_clean,category_final
0,biggie,Alimentos Especiales,alimentos especiales,None
1,biggie,Almacén,almacen,Almacén
2,biggie,Asado,asado,None
3,biggie,Bebes,bebes,Bebes
4,biggie,Bebidas con Alcohol,bebidas con alcohol,Bebidas con Alcohol


In [69]:
df[df["category_final"].isna()][["supermarket", "category_lvl1_name", "category_clean"]].drop_duplicates()

,supermarket,category_lvl1_name,category_clean
0,biggie,Alimentos Especiales,alimentos especiales
2,biggie,Asado,asado
7,biggie,Chocolates y Golosinas,chocolates y golosinas
11,biggie,Heladeria y Confiteria,heladeria y confiteria
12,biggie,Higiene Personal,higiene personal
18,biggie,Snacks,snacks
19,biggie,Varios,varios
351,real,Cuidado del Hogar,cuidado del hogar
522,biggie,None,None
1337,real,Desactivados,desactivados
